# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using the `mlcroissant` library.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print out some basic metadata fields
print(f"Dataset name: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets and their fields by `@id`.

**Note:** In Croissant, a *record set* represents a table-like structure (typically a CSV or similar) containing columns, with each *field* mapped to a column. We use the `@id` to refer to all record sets, fields, and columns.


In [ ]:
# List all record sets in the Croissant metadata using their @id

print("Available record sets (by @id):")
record_sets = []
for rs in getattr(dataset.metadata, 'recordSet', []):
    print(f"- {rs['@id']}")
    record_sets.append(rs['@id'])

# For each record set, list all fields/columns (by @id)
for rs in getattr(dataset.metadata, 'recordSet', []):
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', 'N/A')}")
    print("  Fields (by @id):")
    for field in rs.get('field', []):
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - {field_id}")


## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. All record sets and fields are referenced by their `@id`.

If more than one record set is present, all are loaded into `dataframes` by their `@id`.


In [ ]:
# Load each record set into a pandas DataFrame.
# We'll demonstrate with the first available record set (by @id).

dataframes = {}
if len(record_sets) == 0:
    print("No record sets found in this dataset.")
else:
    for rs_id in record_sets:
        print(f"Loading records for RecordSet @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows, columns: {df.columns.tolist()}")

    # Choose one for further demonstration
    example_record_set = record_sets[0]
    print(f"\nColumns in Example RecordSet ({example_record_set}):\n", dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering, normalizing, basic grouping. All fields are referenced by their `@id` as specified in the Croissant schema. Adjust the `numeric_field_id` and `group_field_id` to match valid columns or field `@id`s in your dataset. Use the output above to identify these.


In [ ]:
# Example: Replace with the actual field/column @id from the chosen record set.

example_df = dataframes[example_record_set]
# You can select any numeric column @id visible in previous output. We'll try to infer a likely numeric column.
numeric_col = None
for col in example_df.columns:
    # Heuristically choose a numeric-looking column
    if 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'score' in col.lower():
        numeric_col = col
        break
if numeric_col is None:
    numeric_col = example_df.select_dtypes(include='number').columns[0] if not example_df.select_dtypes(include='number').empty else example_df.columns[0]

# Set an arbitrary threshold for demo (ensure proper conversion to float if needed)
example_df[numeric_col] = pd.to_numeric(example_df[numeric_col], errors='coerce')
threshold = example_df[numeric_col].mean() if example_df[numeric_col].dtype != object else 0

filtered_df = example_df[example_df[numeric_col] > threshold]
print(f"Filtered records where {numeric_col} > {threshold:.3f} (by @id):")
print(filtered_df.head())

filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
print(f"\nNormalized '{numeric_col}' for filtered records:")
print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

# Now try grouping by another field, e.g. a categorical
group_field = None
for col in example_df.columns:
    if 'group' in col.lower() or 'sample' in col.lower() or 'type' in col.lower():
        group_field = col
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped results by: {group_field} (using @id):")
    print(grouped_df.head())
else:
    print("\nNo obvious group field found for grouping.")

## 5. Visualization

Visualize the numeric field distribution and, if available, group/category statistics. All axes and legend labels display `@id`s, per FAIR principles.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(example_df[numeric_col].dropna(), kde=True)
plt.title(f"Distribution of {numeric_col} (@id)")
plt.xlabel(numeric_col)
plt.ylabel('Frequency')
plt.show()

# Boxplot grouped by group_field (if found)
if group_field:
    plt.figure(figsize=(7,5))
    sns.boxplot(x=group_field, y=numeric_col, data=filtered_df)
    plt.title(f"{numeric_col} by {group_field} (@id)")
    plt.xlabel(group_field)
    plt.ylabel(numeric_col)
    plt.show()

## 6. Conclusion

- We used `mlcroissant` to load metadata and records from the FAIR^2 dataset (Extracellular Vesicle Human Mass Spectrometry).
- The exploration showed available record sets and fields/columns referenced by `@id`.
- Example processing included filtering and normalization of a chosen numeric field, then grouping and visualizing the results.

Continue your analysis by selecting other record sets or fields, and adapting the EDA/visualization to specific scientific questions.
